In [ ]:
# Setup and shared utilities
import json
import os
import random
import statistics
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

# Add project src to path
import sys
sys.path.insert(0, str(project_root / "src"))

from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from context_engineering.config import VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS
from context_engineering.infrastructure.llm_providers import (
    get_chat_llm,
    get_default_embeddings,
 )
from context_engineering.application.chat_service.rag_service import RAGService
from context_engineering.application.chat_service.cag_service import CAGService
from context_engineering.application.chat_service.cag_cache import CAGCache
from context_engineering.application.chat_service.crag_service import CRAGService

# Providers: Groq for chat, OpenRouter for embeddings
llm = get_chat_llm(temperature=0, provider="groq")
embedding_provider = "openrouter"
embeddings = get_default_embeddings(provider=embedding_provider)

# Compact text to reduce embedding cost
MAX_QUERY_CHARS = 600
MAX_ANSWER_CHARS = 1200

def compact_text(text: str, limit: int) -> str:
    normalized = " ".join((text or "").split())
    if len(normalized) > limit:
        return normalized[:limit].rstrip()
    return normalized

def embed_text(text: str) -> np.ndarray:
    return np.array(embeddings.embed_query(compact_text(text, MAX_ANSWER_CHARS)))

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-10
    return float(np.dot(a, b) / denom)

random.seed(42)
np.random.seed(42)

print("Setup complete")

In [ ]:
# Qdrant setup + retrievers per strategy
qdrant_path = VECTOR_DIR / "qdrant"
if not qdrant_path.exists():
    raise FileNotFoundError("Run 02_chunking.ipynb to build vectorstore")

# qdrant-client 1.17+ removed search/search_points; map to query_points
if not hasattr(QdrantClient, "search") and hasattr(QdrantClient, "query_points"):
    def _search_compat(
        self,
        collection_name,
        query_vector,
        limit=10,
        offset=None,
        filter=None,
        params=None,
        with_payload=True,
        with_vectors=False,
        score_threshold=None,
        consistency=None,
        shard_key_selector=None,
        timeout=None,
        **kwargs,
    ):
        # Avoid duplicate kwargs that may already be provided by upstream callers
        kwargs.pop("query_filter", None)
        kwargs.pop("search_params", None)
        kwargs.pop("limit", None)
        kwargs.pop("offset", None)
        kwargs.pop("with_payload", None)
        kwargs.pop("with_vectors", None)
        kwargs.pop("score_threshold", None)
        result = self.query_points(
            collection_name=collection_name,
            query=query_vector,
            using=None,
            prefetch=None,
            query_filter=filter,
            search_params=params,
            limit=limit,
            offset=offset,
            with_payload=with_payload,
            with_vectors=with_vectors,
            score_threshold=score_threshold,
            lookup_from=None,
            consistency=consistency,
            shard_key_selector=shard_key_selector,
            timeout=timeout,
            **kwargs,
        )
        return result.points

    QdrantClient.search = _search_compat

qdrant_client = QdrantClient(path=str(qdrant_path))

strategies = [
    "semantic",
    "fixed",
    "sliding",
    "parent_child_children",
    "late_base",
 ]

def build_retriever(collection_name: str):
    vectorstore = Qdrant(
        client=qdrant_client,
        collection_name=collection_name,
        embeddings=embeddings,
    )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )

retrievers = {name: build_retriever(name) for name in strategies}
rag_services = {name: RAGService(retriever=retrievers[name], llm=llm, k=5) for name in strategies}

print("Retrievers ready:", ", ".join(retrievers.keys()))

In [ ]:
# Chunking strategy comparison (10 queries x 5 strategies)
queries = [
    "What is Primelands and what types of properties are listed?",
    "Which apartments are available in Colombo?",
    "List any land projects in Kaduwatha.",
    "Do you have properties in Gampaha?",
    "What is The Seasons Colombo 08?",
    "Show properties near Kottawa.",
    "Which portfolio properties are in Galle?",
    "Find houses in Kalutara.",
    "Are there listings in Kiribathgoda?",
    "What is the project The Mansion Nawala?",
 ]

def get_top_urls(retriever, query: str, k: int = 5):
    retriever.search_kwargs["k"] = k
    start = time.perf_counter()
    docs = retriever.invoke(query)
    latency = time.perf_counter() - start
    urls = [doc.metadata.get("url", "") for doc in docs if doc.metadata.get("url")]
    return docs, urls, latency

records = []
for raw_query in queries:
    query = compact_text(raw_query, MAX_QUERY_CHARS)
    per_strategy = {}
    url_counts = {}

    # First pass: retrieval only, build consensus relevance set
    for name, retriever in retrievers.items():
        docs, urls, retrieval_latency = get_top_urls(retriever, query, k=5)
        per_strategy[name] = {
            "docs": docs,
            "urls": urls,
            "retrieval_latency": retrieval_latency,
        }
        for url in set(urls):
            url_counts[url] = url_counts.get(url, 0) + 1

    relevant_urls = {url for url, count in url_counts.items() if count >= 2}
    if not relevant_urls:
        relevant_urls = set(url_counts.keys())

    # Second pass: generation and metrics per strategy
    for name in strategies:
        urls = per_strategy[name]["urls"]
        retrieval_latency = per_strategy[name]["retrieval_latency"]
        hits = len(set(urls) & relevant_urls)
        precision_at_5 = hits / 5 if 5 > 0 else 0.0
        recall_at_5 = hits / len(relevant_urls) if relevant_urls else 0.0

        start = time.perf_counter()
        rag_result = rag_services[name].generate(query)
        end_to_end = time.perf_counter() - start

        answer_vec = embed_text(rag_result["answer"])
        query_vec = embed_text(query)
        answer_relevance = cosine_sim(query_vec, answer_vec)

        records.append({
            "query": raw_query,
            "strategy": name,
            "precision_at_5": round(precision_at_5, 4),
            "recall_at_5": round(recall_at_5, 4),
            "answer_relevance": round(answer_relevance, 4),
            "retrieval_latency_ms": round(retrieval_latency * 1000, 2),
            "end_to_end_latency_ms": round(end_to_end * 1000, 2),
        })

# Aggregate results to identify winner
summary = {}
for name in strategies:
    rows = [r for r in records if r["strategy"] == name]
    summary[name] = {
        "precision_at_5": statistics.mean(r["precision_at_5"] for r in rows),
        "recall_at_5": statistics.mean(r["recall_at_5"] for r in rows),
        "answer_relevance": statistics.mean(r["answer_relevance"] for r in rows),
        "end_to_end_latency_ms": statistics.mean(r["end_to_end_latency_ms"] for r in rows),
    }

# Composite score: higher is better, latency is inverted
def composite_score(metrics: dict) -> float:
    inv_latency = 1.0 / (1.0 + (metrics["end_to_end_latency_ms"] / 1000.0))
    return (
        metrics["precision_at_5"]
        + metrics["recall_at_5"]
        + metrics["answer_relevance"]
        + inv_latency
    )

winner = max(summary.items(), key=lambda item: composite_score(item[1]))
print("Chunking winner:", winner[0])
print("Summary:")
for name, metrics in summary.items():
    print(f"  {name}: {metrics}")

# Write output file
csv_path = project_root / "chunking_comparison.csv"
import csv
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
    writer.writeheader()
    writer.writerows(records)

print("Wrote:", csv_path)

In [ ]:
# CAG cache effectiveness (100-query simulation)
cache_dir = CACHE_DIR / "perf_cag_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

cag_cache = CAGCache(
    cache_dir=cache_dir,
    embedder=embeddings,
    similarity_threshold=0.90,
    history_ttl_hours=24,
 )

# Use semantic strategy for baseline RAG
base_rag_service = rag_services["semantic"]
cag_service = CAGService(rag_service=base_rag_service, cache=cag_cache)

base_queries = [
    "What is Primelands?",
    "Which apartments are in Colombo?",
    "Do you have properties in Gampaha?",
    "Show listings in Kottawa.",
    "What is The Seasons Colombo 08?",
    "Are there listings in Kiribathgoda?",
    "Which portfolio properties are in Galle?",
    "Find houses in Kalutara.",
    "Any land projects in Kaduwatha?",
    "What is The Mansion Nawala?",
    "Are there projects in Negombo?",
    "Show apartments in Peliagoda.",
    "List properties in Malabe.",
    "Tell me about Skye Blossom Kottawa.",
    "Which properties are in Athurugiriya?",
    "What is J-Adore Negombo?",
    "List portfolio properties in Anuradhapura.",
    "Do you have land in Ganemulla?",
    "Find properties in Wariyapola.",
    "Show land projects in Kalutara.",
 ]

# Simulate 100 queries with repeats
sim_queries = []
for i in range(100):
    if i < 60:
        sim_queries.append(random.choice(base_queries[:10]))
    else:
        sim_queries.append(random.choice(base_queries))

hit_latencies = []
miss_latencies = []
hits = 0
misses = 0

for q in sim_queries:
    query = compact_text(q, MAX_QUERY_CHARS)
    result = cag_service.generate(query, use_cache=True, verbose=False)
    if result["cache_hit"]:
        hits += 1
        hit_latencies.append(result.get("lookup_time", 0.0))
    else:
        misses += 1
        miss_latencies.append(result.get("generation_time", 0.0))

hit_rate = hits / (hits + misses) if (hits + misses) else 0.0
avg_hit = statistics.mean(hit_latencies) if hit_latencies else 0.0
avg_miss = statistics.mean(miss_latencies) if miss_latencies else 0.0
latency_improvement = (avg_miss - avg_hit) / avg_miss if avg_miss else 0.0

# Cost assumptions (adjust to your provider pricing)
AVG_INPUT_TOKENS = 1200
AVG_OUTPUT_TOKENS = 250
COST_PER_1K_INPUT = 0.0005
COST_PER_1K_OUTPUT = 0.0015
cost_per_call = (AVG_INPUT_TOKENS / 1000) * COST_PER_1K_INPUT + (AVG_OUTPUT_TOKENS / 1000) * COST_PER_1K_OUTPUT
estimated_savings = hits * cost_per_call

cag_stats = {
    "total_queries": len(sim_queries),
    "hits": hits,
    "misses": misses,
    "hit_rate": round(hit_rate, 4),
    "avg_hit_latency_ms": round(avg_hit * 1000, 2),
    "avg_miss_latency_ms": round(avg_miss * 1000, 2),
    "latency_improvement_pct": round(latency_improvement * 100, 2),
    "estimated_cost_savings": round(estimated_savings, 6),
    "assumptions": {
        "avg_input_tokens": AVG_INPUT_TOKENS,
        "avg_output_tokens": AVG_OUTPUT_TOKENS,
        "cost_per_1k_input": COST_PER_1K_INPUT,
        "cost_per_1k_output": COST_PER_1K_OUTPUT,
    },
 }

json_path = project_root / "cag_stats.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(cag_stats, f, indent=2)

print("CAG stats:", cag_stats)
print("Wrote:", json_path)

In [ ]:
# CRAG correction impact (20 queries)
crag_service = CRAGService(retriever=retrievers["semantic"], llm=llm)
rag_service = rag_services["semantic"]

crag_queries = [
    "What is Primelands?",
    "Which apartments are in Colombo?",
    "Do you have properties in Gampaha?",
    "Show listings in Kottawa.",
    "What is The Seasons Colombo 08?",
    "Are there listings in Kiribathgoda?",
    "Which portfolio properties are in Galle?",
    "Find houses in Kalutara.",
    "Any land projects in Kaduwatha?",
    "What is The Mansion Nawala?",
    "Are there projects in Negombo?",
    "Show apartments in Peliagoda.",
    "List properties in Malabe.",
    "Tell me about Skye Blossom Kottawa.",
    "Which properties are in Athurugiriya?",
    "What is J-Adore Negombo?",
    "List portfolio properties in Anuradhapura.",
    "Do you have land in Ganemulla?",
    "Find properties in Wariyapola.",
    "Show land projects in Kalutara.",
 ]

crag_records = []
for q in crag_queries:
    query = compact_text(q, MAX_QUERY_CHARS)
    rag_result = rag_service.generate(query)
    crag_result = crag_service.generate(query, verbose=False)

    query_vec = embed_text(query)
    rag_score = cosine_sim(query_vec, embed_text(rag_result["answer"]))
    crag_score = cosine_sim(query_vec, embed_text(crag_result["answer"]))
    crag_records.append({
        "query": q,
        "rag_confidence": round(crag_result["confidence_initial"], 4),
        "crag_confidence": round(crag_result["confidence_final"], 4),
        "correction_applied": crag_result["correction_applied"],
        "rag_answer_relevance": round(rag_score, 4),
        "crag_answer_relevance": round(crag_score, 4),
        "answer_quality_gain": round(crag_score - rag_score, 4),
    })

corrections = sum(1 for r in crag_records if r["correction_applied"])
avg_conf_gain = statistics.mean(r["crag_confidence"] - r["rag_confidence"] for r in crag_records)
avg_quality_gain = statistics.mean(r["answer_quality_gain"] for r in crag_records)

crag_csv = project_root / "crag_impact.csv"
import csv
with open(crag_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(crag_records[0].keys()))
    writer.writeheader()
    writer.writerows(crag_records)

print(f"Corrections applied: {corrections}/{len(crag_records)}")
print(f"Avg confidence improvement: {avg_conf_gain:.4f}")
print(f"Avg answer quality gain: {avg_quality_gain:.4f}")
print("Wrote:", crag_csv)